# Módulo 10 · Aula 06 — Filas, Mensageria e Orquestração

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O relatório anual demora 4 minutos. O usuário clica, a tela fica branca, ele clica de novo, e agora são dois relatórios rodando. Na terceira vez a API cai."*

E a do pipeline:

> *"O `cron` roda a extração às 3h e a agregação às 3h30. Ontem a extração demorou 40 minutos e a agregação rodou com o dado da véspera. O relatório da manhã estava errado e ninguém sabia."*

Dois problemas, dois assuntos:

| Dor | Resposta |
|-----|----------|
| Tarefa longa numa requisição | **Fila** |
| Etapas com dependência, agendadas por horário | **Orquestrador** |

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Redis como fila | O mais simples que funciona |
| 2 | 🔴 **Trava distribuída** | O `Lock` que funciona entre processos |
| 3 | Celery | A fila com bateria inclusa |
| 4 | 🔴 **Entrega e idempotência** | "Pelo menos uma vez" |
| 5 | RabbitMQ vs Kafka | Fila vs registro de eventos |
| 6 | 🔴 **`cron` não é orquestrador** | Por que o horário não basta |
| 7 | **DAG** | 🎯 Com um executor de verdade |
| 8 | Airflow | Os conceitos que importam |
| 9 | Idempotência de tarefa | O que torna o retry seguro |

> 🎯 **Você vai escrever um executor de DAG** que resolve dependências, repete o que falha e pula o que depende do que quebrou — para entender o que o Airflow faz antes de instalá-lo.

## ⚙️ Preparação

Usamos `fakeredis` (a mesma API do Redis, sem servidor) e o Celery em modo **eager**. Trocar por Redis real é mudar uma linha.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 10
# ═══════════════════════════════════════════════════════════════
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("pandas", "pandas"), ("numpy", "numpy"),
               ("pyarrow", "pyarrow"), ("polars", "polars"),
               ("duckdb", "duckdb")]:
    _garantir(_p, _m)

import numpy as np
import pandas as pd

TEM_POLARS = _garantir("polars", "polars")
TEM_DUCKDB = _garantir("duckdb", "duckdb")
TEM_ARROW = _garantir("pyarrow", "pyarrow")

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"pandas {pd.__version__} · numpy {np.__version__}")
if TEM_POLARS:
    import polars as pl
    print(f"polars {pl.__version__}")
if TEM_DUCKDB:
    import duckdb
    print(f"duckdb {duckdb.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Dados da Aurora — gerados de forma REPRODUTÍVEL
# ═══════════════════════════════════════════════════════════════
SEMENTE = 20260813
rng = np.random.default_rng(SEMENTE)
random.seed(SEMENTE)

CIDADES = ["Campinas", "São Paulo", "Valinhos", "Sumaré",
           "Indaiatuba", "Jundiaí", "Hortolândia"]
CANAIS = ["site", "app", "marketplace"]
CATEGORIAS = {
    "NB": ("Notebooks", 1800, 4200),
    "MO": ("Monitores", 700, 2200),
    "PE": ("Periféricos", 40, 400),
    "AR": ("Armazenamento", 180, 900),
}


def gerar_produtos(n: int = 60) -> pd.DataFrame:
    linhas = []
    for i in range(n):
        prefixo = list(CATEGORIAS)[i % len(CATEGORIAS)]
        categoria, minimo, maximo = CATEGORIAS[prefixo]
        preco = round(float(rng.uniform(minimo, maximo)), 2)
        linhas.append({
            "sku": f"{prefixo}-{1000 + i}",
            "nome": f"{categoria[:-1]} modelo {i:03d}",
            "categoria": categoria,
            "preco": preco,
            "custo": round(preco * float(rng.uniform(0.55, 0.85)), 2),
            "estoque": int(rng.integers(0, 200)),
        })
    return pd.DataFrame(linhas)


def gerar_vendas(n: int = 50_000, dias: int = 180,
                 produtos: pd.DataFrame | None = None) -> pd.DataFrame:
    """Vendas sintéticas com sazonalidade e um pouco de sujeira."""
    produtos = gerar_produtos() if produtos is None else produtos
    fim = datetime(2026, 8, 1, tzinfo=timezone.utc)
    inicio = fim - timedelta(days=dias)

    idx = rng.integers(0, len(produtos), n)
    escolhidos = produtos.iloc[idx].reset_index(drop=True)

    # 📈 Sazonalidade: mais vendas no fim de semana e no fim do mês
    deslocamento = rng.integers(0, dias, n)
    datas = pd.to_datetime(inicio) + pd.to_timedelta(deslocamento, unit="D")
    datas = datas + pd.to_timedelta(rng.integers(0, 86400, n), unit="s")

    return pd.DataFrame({
        "pedido_id": 100_000 + np.arange(n),
        "data": datas,
        "sku": escolhidos["sku"],
        "categoria": escolhidos["categoria"],
        "cidade": rng.choice(CIDADES, n, p=[.28, .22, .12, .12, .11, .09, .06]),
        "canal": rng.choice(CANAIS, n, p=[.55, .30, .15]),
        "quantidade": rng.integers(1, 6, n),
        "preco_unitario": escolhidos["preco"],
        "custo_unitario": escolhidos["custo"],
        "status": rng.choice(["pago", "pendente", "cancelado"], n, p=[.82, .10, .08]),
        "frete": np.round(rng.uniform(0, 45, n), 2),
    })


# ═══════════════════════════════════════════════════════════════
#  Medição
# ═══════════════════════════════════════════════════════════════

def cronometrar(funcao, repeticoes: int = 1):
    """Devolve (resultado, milissegundos_medios)."""
    inicio = time.perf_counter()
    resultado = None
    for _ in range(repeticoes):
        resultado = funcao()
    return resultado, (time.perf_counter() - inicio) * 1000 / repeticoes


def comparar(casos: list[tuple[str, callable]], repeticoes: int = 1,
             rotulo: str = "abordagem"):
    """Mede várias abordagens e mostra o ganho relativo."""
    medidos = []
    for nome, funcao in casos:
        _, ms = cronometrar(funcao, repeticoes)
        medidos.append((nome, ms))
    melhor = min(m for _, m in medidos)
    largura = max(len(n) for n, _ in medidos) + 2
    print(f"{rotulo:<{largura}}{'tempo':>12}   {'vs melhor':>10}")
    print("─" * (largura + 26))
    for nome, ms in medidos:
        barra = "█" * max(1, int(ms / melhor))
        print(f"{nome:<{largura}}{ms:>9.1f} ms   {ms / melhor:>8.1f}×  {barra[:26]}")
    return medidos


def tamanho(n: int) -> str:
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:,.1f} {unidade}" if unidade != "B" else f"{n:,} B"
        n /= 1024
    return ""


def memoria(df: pd.DataFrame) -> int:
    return int(df.memory_usage(deep=True).sum())


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("\n✅ `gerar_vendas()`, `comparar()`, `memoria()`, `tabela()` prontos")
print(f"   semente fixa ({SEMENTE}) — os números são reprodutíveis")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Ferramentas desta aula
# ═══════════════════════════════════════════════════════════════
import threading
from concurrent.futures import ThreadPoolExecutor

for _p, _m in [("fakeredis", "fakeredis"), ("celery", "celery")]:
    _garantir(_p, _m)

TEM_FAKEREDIS = _garantir("fakeredis", "fakeredis")
TEM_CELERY = _garantir("celery", "celery")

BASE = preparar("aula_10_06")


def abrir_fila():
    """Redis real se houver; senão, fakeredis. A API é a mesma."""
    try:
        import redis
        import socket
        with socket.create_connection(("localhost", 6379), timeout=0.4):
            pass
        print("🟥 Redis real em localhost:6379")
        return redis.Redis(decode_responses=True)
    except Exception:
        import fakeredis
        print("📦 fakeredis (em memória) — a mesma API, sem servidor")
        return fakeredis.FakeStrictRedis(decode_responses=True)


fila = abrir_fila()
fila.flushall()
print(f"celery: {'✅' if TEM_CELERY else '⚠️ indisponível'}")

## 1. Redis como fila

In [ ]:
print("""
   O QUE UMA FILA RESOLVE

   ┌────────┐  põe   ┌──────────┐  tira  ┌──────────┐
   │  API   │───────►│   FILA   │◄───────│ WORKER   │
   │ (rápida)│       │(durável) │        │ (lento)  │
   └────────┘        └──────────┘        └──────────┘

   A API responde em milissegundos ("aceitei, id 42").
   O worker leva 4 minutos, noutro processo, sem prender ninguém.

🔑 E a fila DESACOPLA os dois:
   · o worker pode cair e voltar — a mensagem continua lá
   · você pode ter 1 ou 10 workers, sem mudar a API
   · picos de trabalho viram fila, não erro
""")

# A fila mais simples que funciona
def enfileirar(nome: str, tarefa: dict) -> int:
    return fila.lpush(nome, json.dumps(tarefa, ensure_ascii=False))


def desenfileirar(nome: str, espera: int = 1) -> dict | None:
    """🔑 `brpop` BLOQUEIA até chegar algo — não é `while True: sleep`."""
    item = fila.brpop(nome, timeout=espera)
    return json.loads(item[1]) if item else None


for pedido in [9001, 9002, 9003]:
    enfileirar("relatorios", {"tipo": "faturamento", "pedido": pedido})

print(f"na fila: {fila.llen('relatorios')}\n")
while (tarefa := desenfileirar("relatorios")) is not None:
    print(f"   processando {tarefa}")
print(f"\nfila vazia: {fila.llen('relatorios') == 0}")

> 🔑 **`LPUSH` + `BRPOP` é uma fila FIFO com bloqueio — e cabe em duas linhas.**
>
> O `B` de `BRPOP` é *blocking*: o worker dorme até chegar trabalho, sem consumir CPU. Isso já é melhor que o `while True: sleep(1)` que todo mundo escreve primeiro.
>
> 🔴 **Mas há um buraco:** se o worker morrer **depois** do `BRPOP` e **antes** de terminar, a mensagem se perde. Ela já saiu da fila e não foi processada.
>
> A defesa é `BRPOPLPUSH`, que move para uma fila "em processamento" — e um vigia devolve o que ficou lá tempo demais. **Ou você usa uma ferramenta que já faz isso**, que é o assunto da seção 3.

In [ ]:
# 🔑 A fila confiável: BRPOPLPUSH
def desenfileirar_confiavel(origem: str, processando: str, espera: int = 1):
    """Move para 'processando' atomicamente. Se o worker morrer, o item fica lá."""
    item = fila.brpoplpush(origem, processando, timeout=espera)
    return json.loads(item) if item else None


def confirmar(processando: str, tarefa: dict) -> int:
    """Só remove de 'processando' DEPOIS de terminar."""
    return fila.lrem(processando, 1, json.dumps(tarefa, ensure_ascii=False))


fila.flushall()
enfileirar("tarefas", {"id": 1, "acao": "gerar_relatorio"})

t = desenfileirar_confiavel("tarefas", "tarefas:processando")
print(f"   pegou            : {t}")
print(f"   fila principal   : {fila.llen('tarefas')}")
print(f"   em processamento : {fila.llen('tarefas:processando')}  🔑 ainda está lá")

print("\n   ... o worker morre aqui, antes de confirmar ...\n")
print(f"   em processamento : {fila.llen('tarefas:processando')}  "
      f"✅ a tarefa NÃO se perdeu")

confirmar("tarefas:processando", t)
print(f"   após confirmar   : {fila.llen('tarefas:processando')}")

## 2. 🔴 Trava distribuída

In [ ]:
print("""
🔴 O `threading.Lock` DO M04 PROTEGE DENTRO DE UM PROCESSO.

   Com 4 workers do gunicorn (M09) ou 3 réplicas em container (M08),
   são 4 travas independentes protegendo nada.
""")

def travar(chave: str, dono: str, segundos: int = 10) -> bool:
    """`SET chave valor NX EX` — atômico, e é o que faz funcionar.

    🔑 `NX` = grava só se NÃO existir. A atomicidade é do Redis: não há
       janela entre "verificar" e "gravar" para outro worker se meter.
    """
    return bool(fila.set(f"trava:{chave}", dono, nx=True, ex=segundos))


def destravar(chave: str, dono: str) -> bool:
    """🔴 Só destrava se a trava ainda for SUA.

    Sem esta conferência: o worker A trava, demora mais que o TTL, a
    trava expira, o worker B trava — e então A termina e destrava a
    trava DE B. Dois workers passam a rodar ao mesmo tempo.
    """
    script = """
    if redis.call('get', KEYS[1]) == ARGV[1] then
        return redis.call('del', KEYS[1])
    else
        return 0
    end
    """
    try:
        return bool(fila.eval(script, 1, f"trava:{chave}", dono))
    except Exception:
        # fakeredis pode não ter Lua — versão não atômica, só para a aula
        if fila.get(f"trava:{chave}") == dono:
            return bool(fila.delete(f"trava:{chave}"))
        return False


fila.flushall()
print(f"   worker-1 tenta travar: {travar('etl_diario', 'worker-1')}")
print(f"   worker-2 tenta travar: {travar('etl_diario', 'worker-2')}   🔒 barrado")
print(f"   worker-2 tenta destravar (não é dele): {destravar('etl_diario', 'worker-2')}")
print(f"   worker-1 destrava: {destravar('etl_diario', 'worker-1')}")
print(f"   worker-2 tenta de novo: {travar('etl_diario', 'worker-2')}   ✅ agora sim")

In [ ]:
# Provando com concorrência de verdade
fila.flushall()
execucoes = []


def rodar_etl(nome: str):
    if not travar("etl", nome, segundos=5):
        return f"{nome}: 🔒 já está rodando, saindo"
    try:
        execucoes.append(nome)
        time.sleep(0.2)
        return f"{nome}: ✅ executou"
    finally:
        destravar("etl", nome)


with ThreadPoolExecutor(max_workers=5) as executor:
    for resultado in executor.map(rodar_etl, [f"worker-{i}" for i in range(1, 6)]):
        print(f"   {resultado}")

print(f"\n   🎯 5 workers tentaram; {len(execucoes)} executou: {execucoes}")

print("""
💭 É EXATAMENTE O CENÁRIO DA MADRUGADA.

   O agendador dispara o ETL em três máquinas por engano (ou o mesmo
   job atrasa e o próximo começa antes). Sem trava, três pipelines
   escrevem no mesmo lugar ao mesmo tempo.

   ⚠️ E o TTL da trava precisa ser MAIOR que a duração da tarefa. Se o
      ETL leva 40 minutos e a trava expira em 10, você volta a ter o
      problema — com a agravante de achar que está protegido.
""")

## 3. Celery

In [ ]:
if not TEM_CELERY:
    print("⚠️ celery indisponível")
else:
    from celery import Celery

    app = Celery("atlas")
    app.conf.update(
        # 🔑 Em produção: broker="redis://localhost:6379/0"
        task_always_eager=True,        # executa INLINE, para a aula
        task_eager_propagates=True,
        task_acks_late=True,           # 🔑 confirma DEPOIS de terminar
        worker_prefetch_multiplier=1,  # 🔑 não acumula tarefas no worker
        task_reject_on_worker_lost=True,
    )

    @app.task(bind=True, max_retries=3, default_retry_delay=2)
    def gerar_relatorio(self, mes: str) -> dict:
        """Uma tarefa longa, com retry automático."""
        try:
            if mes == "quebrado":
                raise ValueError("mês inválido")
            time.sleep(0.05)
            return {"mes": mes, "linhas": 1420, "gerado_em":
                    datetime.now(timezone.utc).isoformat(timespec="seconds")}
        except ValueError as erro:
            # 🔑 `retry` reenfileira a tarefa; não é um laço no worker
            raise self.retry(exc=erro, countdown=1)

    resultado = gerar_relatorio.delay("2026-07")
    print(f"   .delay() devolve um AsyncResult: {type(resultado).__name__}")
    print(f"   .get()  → {resultado.get()}")

In [ ]:
if TEM_CELERY:
    print("""
🔑 AS QUATRO CONFIGURAÇÕES QUE IMPORTAM

   task_acks_late=True
      Confirma a mensagem DEPOIS de terminar, não ao receber. Se o
      worker morrer no meio, a tarefa volta para a fila.
      🔴 E é isso que exige idempotência: ela pode rodar duas vezes.

   worker_prefetch_multiplier=1
      Sem isso, o worker reserva várias tarefas de uma vez. Se ele
      morrer, TODAS ficam presas até o timeout.

   max_retries + countdown
      Retry com espera — o mesmo backoff do M07.

   task_reject_on_worker_lost=True
      Worker morto devolve a tarefa em vez de perdê-la.

⚠️ E o padrão do Celery é `task_acks_early` — ele confirma ao receber.
   Mais rápido, e perde tarefa se o worker cair. Saiba qual você quer.
""")

    print("Fluxos compostos:\n")
    print("   chain(a.s(), b.s(), c.s())     em sequência, um alimenta o outro")
    print("   group(a.s(), b.s(), c.s())     em paralelo")
    print("   chord(group(...), final.s())   paralelo, e um final ao terminar")
    print("\n   💡 `chord` é o padrão para 'processe os 12 meses e depois")
    print("      consolide' — e é o que o DAG da seção 7 generaliza.")

## 4. 🔴 Entrega e idempotência

In [ ]:
print("""
   AS TRÊS GARANTIAS DE ENTREGA

   no máximo uma vez   pode PERDER · não duplica · rápido
   pelo menos uma vez  não perde · pode DUPLICAR · o padrão prático
   exatamente uma vez  🔶 caríssimo, e frequentemente uma ilusão

🎯 NA PRÁTICA VOCÊ USA "PELO MENOS UMA VEZ" — E TORNA O CONSUMO
   IDEMPOTENTE.

   "Exatamente uma vez" no transporte é quase impossível: se o worker
   confirma antes de terminar, pode perder; se confirma depois, pode
   duplicar. Não há terceira opção quando a rede pode falhar entre as
   duas coisas.

   💭 A solução não está no transporte — está no CONSUMIDOR. Se
      processar duas vezes tem o mesmo efeito que processar uma, a
      duplicação deixa de importar.
""")

# Idempotência na prática — o mesmo padrão do webhook (M07)
fila.flushall()
PROCESSADOS = []


def processar_idempotente(tarefa: dict) -> str:
    """🔑 `SET NX EX` como registro de 'já vi este id'."""
    chave = f"processado:{tarefa['id']}"
    primeira_vez = fila.set(chave, "1", nx=True, ex=86_400)
    if not primeira_vez:
        return f"↩️  {tarefa['id']} já processado, ignorando"
    PROCESSADOS.append(tarefa["id"])
    return f"✅ {tarefa['id']} processado"


tarefa = {"id": "rel-2026-07", "tipo": "relatorio"}
for tentativa in range(1, 4):
    print(f"   entrega {tentativa}: {processar_idempotente(tarefa)}")

print(f"\n   efetivamente processado: {len(PROCESSADOS)}× ✅")

print("""
💭 REPARE QUE É O MESMO CÓDIGO DO WEBHOOK DO M07.

   Lá era o gateway reenviando; aqui é a fila reentregando. O
   mecanismo de defesa é idêntico: uma chave que registra "já vi
   este id", gravada de forma ATÔMICA.

   🎯 Idempotência não é um truque de fila. É uma propriedade que
      você dá ao consumidor — e ela serve para qualquer origem
      duplicada.
""")

## 5. RabbitMQ vs Kafka

In [ ]:
comparacao = [
    ["Metáfora",        "fila de banco",           "livro-caixa"],
    ["Mensagem lida",   "🔑 SOME da fila",         "🔑 FICA no registro"],
    ["Reprocessar",     "não dá",                  "✅ volte o ponteiro"],
    ["Vários leitores", "concorrem pela mesma",    "✅ cada um no seu ritmo"],
    ["Ordem",           "por fila",                "por partição"],
    ["Retenção",        "até consumir",            "por tempo/tamanho"],
    ["Roteamento",      "✅ rico (exchanges)",     "simples (tópico)"],
    ["Vazão",           "alta",                    "🔑 altíssima"],
    ["Use para",        "TAREFAS",                 "EVENTOS"],
]
tabela(["", "RabbitMQ (fila)", "Kafka (registro)"], comparacao, [18, 26, 28])

print("""
🎯 A DIFERENÇA QUE DECIDE: A MENSAGEM SOME OU FICA?

   Numa FILA, ler consome. É o modelo certo para TRABALHO: "gere este
   relatório" precisa ser feito uma vez, por um worker.

   Num REGISTRO, ler não consome — cada consumidor tem seu ponteiro.
   É o modelo certo para EVENTOS: "o pedido 9042 foi pago" interessa
   ao estoque, ao financeiro, ao painel e ao time de dados — cada um
   lendo no seu ritmo, e podendo reprocessar.

💭 E PARA A AURORA, HOJE?

   Nem um nem outro. O Redis com Celery resolve as tarefas, e o
   volume não justifica operar mais um serviço.

   ⚠️ Kafka é excelente e é caro de operar. Adotá-lo antes de precisar
      é o exemplo clássico de arquitetura que consome a equipe.
""")

In [ ]:
# Redis Streams: o meio-termo que quase ninguém conhece
fila.flushall()

for i, evento in enumerate([("venda", "9001"), ("venda", "9002"), ("pagamento", "9001")]):
    fila.xadd("eventos", {"tipo": evento[0], "pedido": evento[1]})

print(f"stream 'eventos': {fila.xlen('eventos')} mensagens\n")

try:
    fila.xgroup_create("eventos", "estoque", id="0")
    fila.xgroup_create("eventos", "financeiro", id="0")

    for grupo in ("estoque", "financeiro"):
        mensagens = fila.xreadgroup(grupo, "consumidor-1", {"eventos": ">"}, count=10)
        n = len(mensagens[0][1]) if mensagens else 0
        print(f"   grupo '{grupo}' leu {n} mensagens")

    print("""
   🔑 OS DOIS GRUPOS LERAM AS MESMAS MENSAGENS.

      É o comportamento do Kafka — cada consumidor no seu ponteiro —
      dentro do Redis que você já tem.

   💡 Redis Streams cobre muito caso de uso de "eventos" sem operar
      mais um serviço. Nem sempre é a resposta, mas quase sempre é a
      resposta que se deve considerar ANTES do Kafka.
""")
except Exception as erro:
    print(f"   ⚠️ grupos de consumidor indisponíveis aqui: {type(erro).__name__}")

## 6. 🔴 `cron` não é orquestrador

In [ ]:
print("""
   O QUE A AURORA TEM HOJE

     0 3 * * *   /opt/atlas/extrair.sh
     30 3 * * *  /opt/atlas/agregar.sh
     0 4 * * *   /opt/atlas/publicar.sh

🔴 E O QUE ELE NÃO SABE

   · a extração terminou?          → o cron dispara às 3h30 de qualquer jeito
   · a extração FALHOU?            → a agregação roda com dado velho
   · alguém precisa ser avisado?   → o cron manda e-mail que ninguém lê
   · dá para reprocessar o dia 12? → você roda o script à mão, torcendo
   · quanto tempo cada etapa levou? → ninguém sabe
   · duas execuções ao mesmo tempo? → o cron não impede

🎯 O `cron` sabe UMA coisa: que horas são.

   Um orquestrador sabe as DEPENDÊNCIAS — e é isso que separa
   "agendado" de "orquestrado".
""")

recursos = [
    ["Dependência entre etapas", "❌", "✅ o DAG"],
    ["Repetir só o que falhou",  "❌", "✅ por tarefa"],
    ["Reprocessar um período",   "❌", "✅ backfill"],
    ["Ver o histórico",          "❌", "✅ interface"],
    ["Impedir sobreposição",     "❌", "✅ max_active_runs"],
    ["Alertar em caso de falha", "🔶 e-mail", "✅ configurável"],
    ["Custo de operação",        "✅ zero", "🔶 mais um serviço"],
]
tabela(["RECURSO", "cron", "ORQUESTRADOR"], recursos, [28, 12, 22])

## 7. 🎯 DAG — com um executor de verdade

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Um executor de DAG — o que o Airflow faz, em 60 linhas
# ═══════════════════════════════════════════════════════════════
from dataclasses import dataclass, field


@dataclass
class Tarefa:
    nome: str
    funcao: callable
    depende_de: list[str] = field(default_factory=list)
    tentativas: int = 1
    espera: float = 0.1


def ordenar(tarefas: dict[str, Tarefa]) -> list[str]:
    """Ordenação topológica — e detecção de ciclo.

    🔑 É o mesmo algoritmo do `needs:` do GitHub Actions (M09). Um DAG
       é um DAG, seja de jobs de CI ou de etapas de ETL.
    """
    pendentes, ordem = dict(tarefas), []
    while pendentes:
        prontas = [n for n, t in pendentes.items()
                   if all(d in ordem for d in t.depende_de)]
        if not prontas:
            raise ValueError(f"🔴 ciclo ou dependência inexistente: {list(pendentes)}")
        for nome in sorted(prontas):
            ordem.append(nome)
            pendentes.pop(nome)
    return ordem


def executar_dag(tarefas: dict[str, Tarefa], contexto: dict) -> dict:
    """Executa na ordem, com retry, e PULA o que depende do que falhou."""
    ordem = ordenar(tarefas)
    print(f"   ordem: {' → '.join(ordem)}\n")

    estado, resultados = {}, {}
    for nome in ordem:
        tarefa = tarefas[nome]

        if any(estado.get(d) in ("falhou", "pulada") for d in tarefa.depende_de):
            estado[nome] = "pulada"
            print(f"   ⏭️  {nome:<22} PULADA (dependência falhou)")
            continue

        for tentativa in range(1, tarefa.tentativas + 1):
            inicio = time.perf_counter()
            try:
                resultados[nome] = tarefa.funcao(contexto, resultados)
                estado[nome] = "sucesso"
                ms = (time.perf_counter() - inicio) * 1000
                sufixo = f"  (tentativa {tentativa})" if tentativa > 1 else ""
                print(f"   ✅ {nome:<22}{ms:>8.0f} ms{sufixo}")
                break
            except Exception as erro:
                if tentativa < tarefa.tentativas:
                    print(f"   🔁 {nome:<22} tentativa {tentativa} falhou "
                          f"({type(erro).__name__}), repetindo")
                    time.sleep(tarefa.espera)
                else:
                    estado[nome] = "falhou"
                    print(f"   🔴 {nome:<22} {type(erro).__name__}: {erro}")

    falhas = [n for n, e in estado.items() if e == "falhou"]
    return {"sucesso": not falhas, "estado": estado,
            "resultados": resultados, "falhas": falhas}


print("✅ executor de DAG pronto")

In [ ]:
# ═══ O DAG do Atlas ═══
LAGO = BASE / "lago"
for c in ("bronze", "prata", "ouro"):
    (LAGO / c).mkdir(parents=True)

ESTADO_ORIGEM = {"instavel": 0}


def extrair_vendas(ctx, res):
    df = gerar_vendas(n=8_000, dias=30)
    caminho = LAGO / "bronze" / f"vendas_{ctx['data']}.parquet"
    df.to_parquet(caminho, index=False)
    return {"linhas": len(df), "caminho": str(caminho)}


def extrair_produtos(ctx, res):
    """Esta origem é instável — falha as 2 primeiras vezes."""
    ESTADO_ORIGEM["instavel"] += 1
    if ESTADO_ORIGEM["instavel"] <= 2:
        raise ConnectionError("origem indisponível")
    df = gerar_produtos()
    caminho = LAGO / "bronze" / f"produtos_{ctx['data']}.parquet"
    df.to_parquet(caminho, index=False)
    return {"linhas": len(df), "caminho": str(caminho)}


def limpar(ctx, res):
    df = pd.read_parquet(res["extrair_vendas"]["caminho"])
    df = df[(df["quantidade"] > 0) & (df["preco_unitario"] > 0)].copy()
    df["receita"] = (df["quantidade"] * df["preco_unitario"]).round(2)
    caminho = LAGO / "prata" / f"vendas_{ctx['data']}.parquet"
    df.to_parquet(caminho, index=False)
    return {"linhas": len(df), "caminho": str(caminho)}


def enriquecer(ctx, res):
    vendas = pd.read_parquet(res["limpar"]["caminho"])
    produtos = pd.read_parquet(res["extrair_produtos"]["caminho"])
    antes = len(vendas)
    juntas = vendas.merge(produtos[["sku", "categoria"]], on="sku",
                          how="left", suffixes=("", "_cad"), validate="many_to_one")
    assert len(juntas) == antes, "🔴 o merge multiplicou linhas"   # M10_02
    caminho = LAGO / "prata" / f"enriquecidas_{ctx['data']}.parquet"
    juntas.to_parquet(caminho, index=False)
    return {"linhas": len(juntas), "caminho": str(caminho)}


def agregar(ctx, res):
    df = pd.read_parquet(res["enriquecer"]["caminho"])
    resumo = (df[df["status"] == "pago"]
              .groupby("categoria", observed=True)
              .agg(pedidos=("pedido_id", "nunique"), receita=("receita", "sum"))
              .reset_index())
    caminho = LAGO / "ouro" / f"faturamento_{ctx['data']}.parquet"
    resumo.to_parquet(caminho, index=False)
    return {"linhas": len(resumo), "caminho": str(caminho)}


def publicar(ctx, res):
    return {"publicado": res["agregar"]["caminho"]}


DAG_ATLAS = {
    "extrair_vendas":   Tarefa("extrair_vendas", extrair_vendas),
    "extrair_produtos": Tarefa("extrair_produtos", extrair_produtos,
                               tentativas=4, espera=0.05),
    "limpar":           Tarefa("limpar", limpar, depende_de=["extrair_vendas"]),
    "enriquecer":       Tarefa("enriquecer", enriquecer,
                               depende_de=["limpar", "extrair_produtos"]),
    "agregar":          Tarefa("agregar", agregar, depende_de=["enriquecer"]),
    "publicar":         Tarefa("publicar", publicar, depende_de=["agregar"]),
}

print("▶ executando o DAG\n")
r = executar_dag(DAG_ATLAS, {"data": "2026-08-13"})
print(f"\n   {'✅ tudo certo' if r['sucesso'] else '🔴 houve falha'}")

> 🎯 **Duas coisas aconteceram aí que o `cron` não faria.**
>
> **1. `extrair_produtos` falhou duas vezes e foi repetida.** Uma indisponibilidade momentânea da origem não derrubou o pipeline — é o retry do M07, aplicado a uma etapa de dados.
>
> **2. `enriquecer` esperou as DUAS dependências.** Ele só rodou quando `limpar` e `extrair_produtos` terminaram — não porque deu 3h30, mas porque as duas ficaram prontas.
>
> 💭 **E repare no `assert` dentro de `enriquecer`:** é o `validate="many_to_one"` da aula 10_02 dentro de uma tarefa de pipeline. A multiplicação silenciosa de linhas seria descoberta aqui, não no relatório.

In [ ]:
# 🔴 E quando uma etapa falha de verdade?
print("▶ mesma DAG, mas a origem de produtos está fora há horas\n")

ESTADO_ORIGEM["instavel"] = -100          # nunca vai se recuperar
DAG_FALHA = dict(DAG_ATLAS)
DAG_FALHA["extrair_produtos"] = Tarefa("extrair_produtos", extrair_produtos,
                                       tentativas=2, espera=0.05)

r2 = executar_dag(DAG_FALHA, {"data": "2026-08-14"})
print(f"\n   {'✅' if r2['sucesso'] else '🔴'} falhas: {r2['falhas']}")

print("""
🎯 REPARE NO QUE ACONTECEU:

   · `extrair_vendas` e `limpar` RODARAM — eles não dependem da origem
     que caiu
   · `enriquecer`, `agregar` e `publicar` foram PULADAS
   · nada foi publicado com dado incompleto

   💭 O `cron` teria rodado a agregação de qualquer forma, às 3h30 —
      com o produto da véspera, ou sem produto nenhum. E o relatório
      da manhã estaria errado sem ninguém saber.
""")

In [ ]:
# Ciclo: o erro de desenho
try:
    ordenar({
        "a": Tarefa("a", lambda c, r: None, depende_de=["c"]),
        "b": Tarefa("b", lambda c, r: None, depende_de=["a"]),
        "c": Tarefa("c", lambda c, r: None, depende_de=["b"]),
    })
except ValueError as erro:
    print(f"   {erro}")

print("""
🔑 O 'A' DE DAG É DE ACÍCLICO — e não é detalhe.

   Se A depende de B que depende de A, não existe ordem possível. O
   ciclo é sempre erro de DESENHO: uma das dependências não é de
   ordem, e sim de dado que deveria vir de outro lugar.
""")

## 8. Airflow — os conceitos que importam

In [ ]:
DAG_ARQUIVO = BASE / "atlas_diario.py"
DAG_ARQUIVO.write_text('''"""DAG do Atlas — referência de Airflow.

⚠️ Este arquivo NÃO roda aqui (exige Airflow instalado e um agendador).
   Ele existe para você comparar com o executor que acabou de escrever:
   os conceitos são os mesmos, com nomes diferentes.
"""
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.python import PythonOperator

argumentos_padrao = {
    "owner": "engenharia-dados",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,      # 🔑 o backoff do M07
    "email_on_failure": True,
}

with DAG(
    dag_id="atlas_diario",
    default_args=argumentos_padrao,
    description="ETL diário do Atlas",
    schedule="0 3 * * *",
    start_date=datetime(2026, 1, 1),

    # 🔴 catchup=False: sem isto, ao ligar o DAG hoje o Airflow tenta
    #    executar TODOS os dias desde o start_date. Já derrubou muito
    #    banco de produção — inclusive o de quem sabia disso.
    catchup=False,

    # 🔑 impede que a execução de hoje comece antes de a de ontem
    #    terminar — o problema que o cron não resolve
    max_active_runs=1,

    tags=["atlas", "diario"],
) as dag:

    def _extrair(**contexto):
        # 🔑 `ds` é a DATA LÓGICA da execução, não "hoje".
        #    É isso que torna o backfill possível: reprocessar o dia 12
        #    passa ds="2026-08-12", e a tarefa faz o que faria naquele dia.
        data = contexto["ds"]
        ...

    extrair = PythonOperator(task_id="extrair", python_callable=_extrair)
    limpar = PythonOperator(task_id="limpar", python_callable=lambda **c: ...)
    agregar = PythonOperator(task_id="agregar", python_callable=lambda **c: ...)

    # 🔑 As dependências, declaradas
    extrair >> limpar >> agregar
''', encoding="utf-8")

print(DAG_ARQUIVO.read_text(encoding="utf-8")[:1400])

In [ ]:
print("""
🔑 OS CONCEITOS QUE VOCÊ JÁ CONSTRUIU

   Airflow                 o que você escreveu
   ───────────────────────────────────────────────────
   DAG                     o dicionário de Tarefas
   Task / Operator         a Tarefa
   a >> b                  depende_de=["a"]
   retries                 tentativas
   upstream_failed         "pulada"
   ds (data lógica)        contexto["data"]

🎯 E OS TRÊS QUE VALEM MAIS QUE O RESTO

   1. catchup=False
      🔴 Ligar um DAG com start_date antigo e catchup ligado dispara
         centenas de execuções de uma vez.

   2. max_active_runs=1
      Impede que a execução de hoje comece antes de ontem terminar.
      É o problema que a trava distribuída resolve à mão.

   3. ds — a data LÓGICA
      🎯 A tarefa recebe a data que ela deveria processar, não "hoje".
         É o que torna o BACKFILL possível: reprocessar o dia 12 é
         rodar com ds="2026-08-12".

   💭 Se a sua tarefa usa `date.today()` por dentro, ela NÃO é
      reprocessável — e o backfill produz dados errados em silêncio.
""")

In [ ]:
# 🔴 A diferença entre tarefa reprocessável e não reprocessável
print("🔴 NÃO reprocessável:\n")
print("   def extrair():")
print("       hoje = date.today()                      # 🔴 sempre hoje")
print("       df = consultar(f'WHERE data = {hoje}')")
print("       gravar(f'bronze/data={hoje}/')")
print("\n   Reprocessar o dia 12 grava em 'data=hoje' com dados de hoje.\n")

print("✅ Reprocessável:\n")
print("   def extrair(data_logica):")
print("       df = consultar(f'WHERE data = {data_logica}')")
print("       gravar(f'bronze/data={data_logica}/')")
print("\n   Reprocessar o dia 12 lê o dia 12 e grava no dia 12.")

print("""
🎯 A REGRA: TODA TAREFA RECEBE A DATA COMO PARÂMETRO.

   Nenhum `date.today()`, `datetime.now()` ou `NOW()` no SQL dentro da
   lógica de negócio da tarefa.

   💭 É o mesmo princípio de injetar dependência do M06: o que vem de
      fora pode ser substituído — inclusive pelo passado.
""")

## 9. Backfill — reprocessar o passado

In [ ]:
def executar_periodo(dag: dict, de: date, ate: date) -> pd.DataFrame:
    """Backfill: roda o DAG para cada dia do período.

    🔑 Funciona porque toda tarefa recebe `ctx["data"]` — nenhuma
       consulta o relógio.
    """
    linhas = []
    dia = de
    while dia <= ate:
        ESTADO_ORIGEM["instavel"] = 99          # origem estável agora
        resultado = executar_dag(dag, {"data": str(dia)})
        linhas.append({"data": str(dia),
                       "sucesso": resultado["sucesso"],
                       "tarefas_ok": sum(1 for e in resultado["estado"].values()
                                         if e == "sucesso")})
        dia += timedelta(days=1)
    return pd.DataFrame(linhas)


print("▶ backfill de 3 dias\n")
historico = executar_periodo(DAG_ATLAS, date(2026, 8, 10), date(2026, 8, 12))
print(f"\n{historico.to_string(index=False)}")

arquivos = sorted((LAGO / "ouro").glob("*.parquet"))
print(f"\n   {len(arquivos)} arquivos na camada ouro:")
for a in arquivos:
    print(f"      {a.name}")

print("""
   🎯 Cada dia gerou o SEU arquivo, com o SEU nome.

   💭 Se as tarefas usassem `date.today()`, os três backfills teriam
      sobrescrito o mesmo arquivo — e você teria três execuções e um
      resultado só.
""")

## 🔧 Prática guiada — o que a Aurora deveria adotar

In [ ]:
escolhas = [
    ["Relatório sob demanda (4 min)", "fila (Celery + Redis)",
     "a API responde em ms"],
    ["ETL diário",                    "orquestrador",
     "há dependência entre etapas"],
    ["Envio de e-mail",               "fila",
     "não pode prender a requisição"],
    ["Webhook do gateway (M07)",      "fila",
     "responda 202 e processe depois"],
    ["Eventos para vários consumidores", "Redis Streams",
     "antes de considerar Kafka"],
    ["Limpeza semanal",               "cron basta",
     "🔑 uma etapa, sem dependência"],
]
tabela(["CASO", "FERRAMENTA", "POR QUÊ"], escolhas, [32, 24, 28])

print("""
💭 E A ÚLTIMA LINHA É A MAIS IMPORTANTE.

   Para UMA tarefa sem dependência, o `cron` continua sendo a resposta
   certa. Ele é confiável, universal e custa zero.

   🎯 O orquestrador entra quando há DEPENDÊNCIA e REPROCESSAMENTO —
      não quando há agendamento.

⚠️ E o Airflow tem um custo real: um banco, um agendador, um servidor
   web, e a curva de aprendizado. Para a Aurora, hoje, um script com
   trava distribuída e um `cron` bem escrito resolve — e o executor de
   DAG desta aula é a ponte quando isso deixar de bastar.
""")

In [ ]:
# Um agendador simples e honesto — o passo intermediário
AGENDADOR = BASE / "rodar_etl.py"
AGENDADOR.write_text('''#!/usr/bin/env python3
"""ETL diário do Atlas — o passo antes do Airflow.

    0 3 * * *  /opt/atlas/.venv/bin/python /opt/atlas/scripts/rodar_etl.py

🔑 O que este script tem que o `cron` sozinho não tem:
   · trava distribuída (não roda duas vezes)
   · data lógica como parâmetro (reprocessável)
   · registro estruturado (M04) e código de saída (M09)
"""
import sys
from datetime import date, timedelta

TRAVA = "etl_diario"


def main() -> int:
    # 🔑 A data vem de fora — reprocessar é `rodar_etl.py 2026-08-12`
    data = date.fromisoformat(sys.argv[1]) if len(sys.argv) > 1 else (
        date.today() - timedelta(days=1))

    if not travar(TRAVA, dono=f"pid-{os.getpid()}", segundos=7200):
        print("🔒 já está rodando — saindo sem erro")
        return 0        # 🔑 não é falha: é o comportamento desejado

    try:
        resultado = executar_dag(DAG_ATLAS, {"data": str(data)})
        return 0 if resultado["sucesso"] else 1
    finally:
        destravar(TRAVA, dono=f"pid-{os.getpid()}")


if __name__ == "__main__":
    sys.exit(main())
''', encoding="utf-8")

print(AGENDADOR.read_text(encoding="utf-8"))

> 🔑 **O `return 0` quando a trava está ocupada é uma decisão, não um descuido.**
>
> Se a execução anterior ainda está rodando, sair sem erro é o comportamento certo — não há nada errado, só há trabalho em andamento. Retornar erro faria o monitoramento (M09) alertar todo dia sem motivo, e em duas semanas ninguém olharia mais os alertas.
>
> 💭 **É a fadiga de alerta do M09 aplicada ao pipeline.** Um alerta que dispara sem exigir ação ensina a equipe a ignorar todos.

## 📝 Exercícios

**E1.** Implemente uma fila com `LPUSH`/`BRPOP` e um worker que consome até a fila esvaziar.

**E2.** 🔴 Mostre a mensagem se perdendo quando o worker morre após o `BRPOP`. Corrija com `BRPOPLPUSH`.

**E3.** 🔴 Implemente a trava distribuída com `SET NX EX`. Prove com 5 threads que só uma executa.

**E4.** Explique por que `destravar` precisa conferir o dono. Descreva o cenário em que a falta disso quebra.

**E5.** Configure o Celery com `acks_late` e explique o que isso exige do consumidor.

**E6.** Implemente uma tarefa com retry e backoff. Provoque duas falhas e veja recuperar.

**E7.** 🔴 Implemente o consumo idempotente com `SET NX EX`. Entregue a mesma tarefa três vezes.

**E8.** Explique por que "exatamente uma vez" é quase sempre uma ilusão no transporte.

**E9.** Compare RabbitMQ e Kafka. Dê um caso do Atlas para cada.

**E10.** Use Redis Streams com dois grupos de consumidor e mostre os dois lendo as mesmas mensagens.

**E11.** 🔴 Liste seis coisas que o `cron` não sabe e diga qual o orquestrador resolve.

**E12.** 🎯 Escreva um executor de DAG com ordenação topológica, retry e propagação de falha.

**E13.** Provoque um ciclo no DAG e mostre a detecção.

**E14.** 🔴 Faça uma etapa falhar e prove que as dependentes são puladas, não executadas com dado velho.

**E15.** 🎯 Reescreva uma tarefa que usa `date.today()` para receber a data como parâmetro. Explique o que isso habilita.

**E16.** Implemente um backfill de 5 dias e mostre um arquivo por dia.

**E17.** Explique `catchup=False` e o que acontece sem ele.

**E18.** Escreva o script agendado com trava, data como parâmetro e código de saída. Justifique o `return 0` na trava ocupada.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

In [ ]:
# E17

In [ ]:
# E18

## 📋 Cola de referência

```python
# ═══ Fila com Redis ═══
r.lpush("fila", json.dumps(tarefa))          # põe
r.brpop("fila", timeout=5)                   # tira, BLOQUEANDO
r.brpoplpush("fila", "processando", 5)       # 🔑 não perde se o worker morrer
r.lrem("processando", 1, item)               # confirma DEPOIS de terminar

# ═══ 🔴 Trava distribuída ═══
r.set(f"trava:{k}", dono, nx=True, ex=600)   # 🔑 NX = só se não existir
# destravar: confira o DONO antes (Lua/compare-and-delete)
# ⚠️ TTL > duração da tarefa

# ═══ Celery ═══
app.conf.task_acks_late = True               # confirma DEPOIS
app.conf.worker_prefetch_multiplier = 1      # não acumula
@app.task(bind=True, max_retries=3)
def t(self, x):
    try: ...
    except E as e: raise self.retry(exc=e, countdown=5)
chain · group · chord

# ═══ 🔴 Entrega ═══
# "pelo menos uma vez" é o padrão prático → torne o CONSUMO idempotente
if r.set(f"visto:{id}", "1", nx=True, ex=86400) is None: return  # já processado

# ═══ Fila vs registro ═══
# RabbitMQ  ler CONSOME   → TAREFAS
# Kafka     ler NÃO consome → EVENTOS (vários leitores, reprocessável)
# Redis Streams: o meio-termo que você já tem instalado

# ═══ 🎯 DAG ═══
# ordenação topológica · retry por tarefa · dependente de falha = PULADA
# ciclo = erro de DESENHO

# ═══ Airflow ═══
catchup=False          # 🔴 senão dispara todo o histórico de uma vez
max_active_runs=1      # não sobrepõe execuções
ds                     # 🎯 a data LÓGICA — é o que permite backfill
# 🔴 nenhum date.today() dentro da tarefa

# ═══ Quando usar o quê ═══
# 1 tarefa, sem dependência   → cron basta
# tarefa longa numa requisição → fila
# etapas com dependência       → orquestrador
```

## ✅ Checklist de saída

**Filas**

- [ ] Sei montar uma fila com `LPUSH`/`BRPOP`
- [ ] 🔴 **Sei que `BRPOP` perde a mensagem se o worker morrer**
- [ ] Uso `BRPOPLPUSH` + confirmação explícita

**Concorrência**

- [ ] 🔴 **Sei que `threading.Lock` não protege entre processos**
- [ ] Implemento trava distribuída com `SET NX EX`
- [ ] Confiro o dono antes de destravar
- [ ] O TTL é maior que a tarefa

**Entrega**

- [ ] Conheço as três garantias
- [ ] 🎯 **Torno o consumo idempotente**
- [ ] Sei que é o mesmo padrão do webhook (M07)
- [ ] Sei o que `acks_late` exige

**Mensageria**

- [ ] Sei a diferença entre fila e registro de eventos
- [ ] Considero Redis Streams antes do Kafka
- [ ] 💭 **Sei argumentar que a Aurora não precisa de Kafka**

**Orquestração**

- [ ] 🔴 **Sei o que o `cron` não sabe**
- [ ] 🎯 **Sei escrever um executor de DAG**
- [ ] Falha propaga como "pulada", não como execução com dado velho
- [ ] Detecto ciclos
- [ ] Sei o que `catchup=False` e `max_active_runs` fazem
- [ ] 🎯 **Toda tarefa minha recebe a data como parâmetro**
- [ ] Sei fazer backfill
- [ ] Sei quando o `cron` ainda é a resposta certa

---

### ➡️ Próxima aula

**`10_99_Lista_Exercicios.ipynb`** — A lista do módulo e o projeto final: o pipeline de dados do Atlas, de ponta a ponta. E o fechamento do manual.